In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import shutil
import random

dataset_root = "/content/drive/MyDrive/Antika_Data_V2"
output_root = "/content/artifact_dataset_split"

for cls in os.listdir(dataset_root):
    cls_path = os.path.join(dataset_root, cls)
    images = os.listdir(cls_path)
    random.shuffle(images)

    n = len(images)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    for i, img in enumerate(images):
        src = os.path.join(cls_path, img)
        if i < train_end:
            dst = os.path.join(output_root, "train", cls, img)
        elif i < val_end:
            dst = os.path.join(output_root, "val", cls, img)
        else:
            dst = os.path.join(output_root, "test", cls, img)

        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)

print("Dataset split completed!")


Dataset split completed!


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "/content/artifact_dataset_split/train"
val_dir = "/content/artifact_dataset_split/val"

# Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Only rescaling for validation
val_datagen = ImageDataGenerator(rescale=1./255)

# Flow from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)


Found 1901 images belonging to 5 classes.
Found 407 images belonging to 5 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# Load base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base model
base_model.trainable = False

# Add custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(5, activation='softmax')(x)  # 5 classes

model = Model(inputs=base_model.input, outputs=predictions)

# Compile
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
epochs = 10

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.5869 - loss: 1.0802 - val_accuracy: 0.7666 - val_loss: 0.5964
Epoch 2/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - accuracy: 0.8124 - loss: 0.4821 - val_accuracy: 0.8010 - val_loss: 0.5514
Epoch 3/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.8465 - loss: 0.4143 - val_accuracy: 0.8526 - val_loss: 0.4731
Epoch 4/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - accuracy: 0.8750 - loss: 0.3382 - val_accuracy: 0.8428 - val_loss: 0.5270
Epoch 5/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.8907 - loss: 0.3258 - val_accuracy: 0.8084 - val_loss: 0.5143
Epoch 6/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.8844 - loss: 0.3225 - val_accuracy: 0.8157 - val_loss: 0.4834
Epoch 7/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - accuracy: 0.9035 - loss: 0.2648 - val_accuracy: 0.8428 - val_loss: 0.4656
Epoch 8/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.8964 - loss: 0.2516 - val_accuracy: 0.8182 - v

In [ ]:
test_dir = "/content/artifact_dataset_split/test"
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

loss, acc = model.evaluate(test_generator)
print(f"Test Accuracy: {acc*100:.2f}%")


Found 412 images belonging to 5 classes.
13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.8034 - loss: 0.6164
Test Accuracy: 76.46%


In [ ]:
# Unfreeze the last 50 layers of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Compile again with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Optional: Early stopping and checkpoint
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('best_finetuned_model.h5', save_best_only=True)
]

# Train for 5–10 epochs (fine-tuning usually needs fewer epochs)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,        # start with 5–10 epochs
    callbacks=callbacks
)


Epoch 1/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7761 - loss: 0.6541

60/60 ━━━━━━━━━━━━━━━━━━━━ 170s 3s/step - accuracy: 0.7769 - loss: 0.6513 - val_accuracy: 0.7887 - val_loss: 0.6593
Epoch 2/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8807 - loss: 0.3009

60/60 ━━━━━━━━━━━━━━━━━━━━ 153s 3s/step - accuracy: 0.8807 - loss: 0.3007 - val_accuracy: 0.8059 - val_loss: 0.6107
Epoch 3/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9213 - loss: 0.2091

60/60 ━━━━━━━━━━━━━━━━━━━━ 157s 3s/step - accuracy: 0.9211 - loss: 0.2096 - val_accuracy: 0.8084 - val_loss: 0.6045
Epoch 4/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9099 - loss: 0.2207

60/60 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.9098 - loss: 0.2208 - val_accuracy: 0.8084 - val_loss: 0.5853
Epoch 5/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.9286 - loss: 0.1751 - val_accuracy: 0.8256 - val_loss: 0.5874
Epoch 6/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.9363 - loss: 0.1662 - val_accuracy: 0.8305 - val_loss: 0.6124
Epoch 7/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.9308 - loss: 0.1535 - val_accuracy: 0.8059 - val_loss: 0.6248


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Path to your test set
test_dir = "/content/artifact_dataset_split/test"

# Only rescale images
test_datagen = ImageDataGenerator(rescale=1./255)

# Create test generator
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # important for evaluation
)

# Evaluate the fine-tuned model
loss, acc = model.evaluate(test_generator)
print(f"Test Accuracy after fine-tuning: {acc*100:.2f}%")
print(f"Test Loss after fine-tuning: {loss:.4f}")



Found 412 images belonging to 5 classes.
13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.8268 - loss: 0.6190
Test Accuracy after fine-tuning: 79.13%
Test Loss after fine-tuning: 0.7221


In [ ]:
model.save("/content/fine_tuned_artifact_model.keras")


In [ ]:
!cp /content/fine_tuned_artifact_model.keras /content/drive/MyDrive/


In [ ]:
train_generator.class_indices



{'Egyptian coin': 0,
 'Egyptian jewelry': 1,
 'Egyptian pottery': 2,
 'Egyptian relief': 3,
 'Egyptian statue': 4}